In [23]:
import pandas as pd
import glob
import os
import re

In [24]:
FOLDER_PATH = 'data_mentah'
TOPIC_LIST_FILE = 'udemy_list_topics.txt'
OUTPUT_FILE = 'udemy_course_dataset.csv'

In [25]:
if not os.path.exists(FOLDER_PATH):
    print(f"Error: Folder '{FOLDER_PATH}' tidak ditemukan.")
    print("Pastikan folder tersebut berada di direktori yang sama dengan notebook ini.")
    file_list = [] 
else:
    file_list = glob.glob(os.path.join(FOLDER_PATH, "*.csv"))
    
    if not file_list:
        print(f"Tidak ada file CSV yang ditemukan di folder '{FOLDER_PATH}'.")
    else:
        print(f"Ditemukan {len(file_list)} file CSV yang siap dianalisis.")

Ditemukan 102 file CSV yang siap dianalisis.


In [26]:
df_list = []

if file_list:    
    for file in file_list:
        try:
            df = pd.read_csv(file)
            filename = os.path.basename(file)
            n_rows, n_cols = df.shape
            missing_values = df.isnull().sum()
            
            print(f"File     : {filename}")
            print(f"Dimensi  : {n_rows} baris, {n_cols} kolom")
            
            total_missing = missing_values.sum()
            if total_missing == 0:
                print("Missing  : 0 missing value secara keseluruhan")
            else:
                print("Missing Value per Kolom:")
                for col, missing in missing_values.items():
                    if missing > 0:
                        print(f"    {col}: {missing} missing value(s)")
            print("-" * 60)
            
            df_list.append(df)
            
        except Exception as e:
            print(f"Gagal membaca file {os.path.basename(file)}: {e}")
            print("-" * 60)
    
    print(f"\nSelesai! Sebanyak {len(df_list)} file telah dibaca dan dianalisis.")
else:
    print("Tidak ada file untuk dianalisis.")

File     : udemy_dataset_course_2D_Game_Development.csv
Dimensi  : 961 baris, 16 kolom
Missing Value per Kolom:
    durations: 9 missing value(s)
    ratings: 3 missing value(s)
    num_ratings: 22 missing value(s)
    student: 7 missing value(s)
------------------------------------------------------------
File     : udemy_dataset_course_3D_Game_Development.csv
Dimensi  : 931 baris, 16 kolom
Missing Value per Kolom:
    durations: 14 missing value(s)
    ratings: 4 missing value(s)
    num_ratings: 20 missing value(s)
    student: 5 missing value(s)
    price: 1 missing value(s)
------------------------------------------------------------
File     : udemy_dataset_course_3D_Modeling.csv
Dimensi  : 981 baris, 16 kolom
Missing Value per Kolom:
    durations: 6 missing value(s)
    ratings: 3 missing value(s)
    num_ratings: 32 missing value(s)
    student: 3 missing value(s)
    last_update: 3 missing value(s)
------------------------------------------------------------
File     : udemy_

In [27]:
if len(df_list) > 0:
    df_gabungan = pd.concat(df_list, ignore_index=True)
    print(f"Berhasil menggabungkan {len(df_list)} file CSV.")
    print(f"   Dimensi sebelum pembersihan: {df_gabungan.shape[0]} baris, {df_gabungan.shape[1]} kolom")
    
    df_gabungan.dropna(inplace=True)
    print(f"   Dimensi setelah pembersihan: {df_gabungan.shape[0]} baris, {df_gabungan.shape[1]} kolom\n")
else:
    print("❌ df_list kosong, pastikan cell sebelumnya sudah dijalankan.")
    exit()

def parse_duration_to_minutes(val):
    if pd.isna(val):
        return None
    val = str(val).strip().lower()
    match = re.match(r'([\d.]+)\s*(hour|hours|min|mins)', val)
    if match:
        number = float(match.group(1))
        unit   = match.group(2)
        if 'hour' in unit:
            return int(round(number * 60))
        else:
            return int(round(number))
    return None

df_gabungan['durations'] = df_gabungan['durations'].apply(parse_duration_to_minutes)
print("Kolom 'durations' berhasil dikonversi ke menit.")
print(df_gabungan['durations'].describe())

level_map = {
    'Beginner'     : 1,
    'Intermediate' : 2,
    'Expert'       : 3,
    'All Levels'   : 4,
}
df_gabungan['level'] = df_gabungan['level'].map(level_map)
print("\nKolom 'level' berhasil di-encode.")
print("   1 = Beginner | 2 = Intermediate | 3 = Expert | 4 = All Levels")
print(df_gabungan['level'].value_counts().sort_index())

topic_map = {}
with open(TOPIC_LIST_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        m = re.match(r'^(\d+)\.\s+(.+)$', line)
        if m:
            topic_map[m.group(2).strip()] = int(m.group(1))

unmapped = df_gabungan[~df_gabungan['subject'].isin(topic_map)]['subject'].unique()
if len(unmapped) > 0:
    print(f"\n  Subjek tidak ditemukan di topic list (akan menjadi NaN): {unmapped}")

df_gabungan['subject'] = df_gabungan['subject'].map(topic_map)
print("\nKolom 'subject' berhasil di-encode sesuai nomor urut udemy_list_topics.txt.")
print(df_gabungan['subject'].describe())

df_gabungan['course_id'] = df_gabungan['course_id'].astype('Int64')
print("\nKolom 'course_id' berhasil dikonversi ke integer.")
print(df_gabungan['course_id'].head(10).to_string())

if 'num_ratings' in df_gabungan.columns:
    df_gabungan['num_ratings'] = df_gabungan['num_ratings'].astype(str).str.strip()
    df_gabungan.loc[df_gabungan['num_ratings'].str.lower() == 'nan', 'num_ratings'] = pd.NA
    df_gabungan['num_ratings'] = df_gabungan['num_ratings'].str.replace(r'[.,]', '', regex=True)
    df_gabungan['num_ratings'] = pd.to_numeric(df_gabungan['num_ratings'], errors='coerce').astype('Int64')

if 'student' in df_gabungan.columns:
    df_gabungan['student'] = pd.to_numeric(df_gabungan['student'], errors='coerce').astype('Int64')

def clean_price(val):
    if pd.isna(val):
        return None
    val = str(val).strip().lower()
    if val == 'free':
        return 0
    val = val.replace('rp', '').strip()
    val = val.replace(',', '')
    val = val.replace('.', '')
    try:
        return int(float(val))
    except Exception:
        return None

if 'price' in df_gabungan.columns:
    df_gabungan['price'] = df_gabungan['price'].apply(clean_price).astype('Int64')

for col in ['num_ratings', 'student', 'price']:
    if col in df_gabungan.columns:
        df_gabungan[col] = df_gabungan[col].fillna(0).astype('int64')

print("\nKolom 'num_ratings', 'student', dan 'price' berhasil dikonversi ke integer (no floats).")

print("\n" + "="*60)
print("HASIL TRANSFORMASI DATA")
print("="*60)
print("Tipe data setelah transformasi:")
cols_to_show = [c for c in ['course_id', 'level', 'durations', 'subject', 'num_ratings', 'student', 'price'] if c in df_gabungan.columns]
print(df_gabungan[cols_to_show].dtypes)
print("\nContoh 5 baris pertama:")
print(df_gabungan[[c for c in ['course_id', 'subject', 'level', 'durations', 'num_ratings', 'student', 'price'] if c in df_gabungan.columns]].head())

df_gabungan.to_csv(OUTPUT_FILE, index=False)
print(f"\nData hasil transformasi disimpan ke: {OUTPUT_FILE}")
print(f"   Dimensi akhir: {df_gabungan.shape[0]} baris, {df_gabungan.shape[1]} kolom")

Berhasil menggabungkan 102 file CSV.
   Dimensi sebelum pembersihan: 95700 baris, 16 kolom
   Dimensi setelah pembersihan: 83146 baris, 16 kolom

Kolom 'durations' berhasil dikonversi ke menit.
count    83146.000000
mean       420.253746
std        676.923683
min          1.000000
25%         90.000000
50%        210.000000
75%        450.000000
max      17400.000000
Name: durations, dtype: float64

Kolom 'level' berhasil di-encode.
   1 = Beginner | 2 = Intermediate | 3 = Expert | 4 = All Levels
level
1    26467
2     9230
3     1255
4    46194
Name: count, dtype: int64

Kolom 'subject' berhasil di-encode sesuai nomor urut udemy_list_topics.txt.
count    83146.000000
mean        57.254552
std         45.917910
min          1.000000
25%         30.000000
50%         54.000000
75%         81.000000
max        405.000000
Name: subject, dtype: float64

Kolom 'course_id' berhasil dikonversi ke integer.
0     258316
1    5348848
2    5258518
3    5059176
4    6293523
5    5618224
6    11781